# Module 9 project-exercise demo — Corpus decisions and state sharding

**Implementation demo, not a completed full-scale project.** Demo 1 runs filters and exact shingle comparisons on synthetic documents with supplied quality labels. Demo 2 simulates shard communication and updates on CPU. These prepare the projects; they do not implement MinHash, a classifier, real FSDP, or GPU measurements.
All inputs are small, deterministic, offline, and CPU-friendly. The goal is to
validate interfaces, metrics, invariants, and report structure before spending
GPU or API budget.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"reproducible smoke tier; seed={SEED}")
DEMO_LANGUAGE = 'en'

plt.rcParams.update({"font.family":"DejaVu Sans", "font.size":12, "axes.titlesize":15, "axes.labelsize":13, "xtick.labelsize":11, "ytick.labelsize":11, "figure.dpi":150})


## Demo 1. Trace a document’s rejection

Eight synthetic documents include a case-normalized duplicate, a near-duplicate, repetitive text, and a fabricated email address. Language and quality labels are supplied. A document is assigned to its first failing stage, so a stage’s count depends on the filters before it. Deduplication compares only against earlier retained documents.

In [ ]:
import re
from collections import Counter
# Synthetic documents; language and quality labels are supplied, not inferred.
docs=[
 ("a","en",.9,"matrix updates use gradients to change model weights"),
 ("b","en",.8,"MATRIX updates use gradients to change model weights"),
 ("c","en",.85,"matrix updates use gradients to change neural weights"),
 ("d","en",.1,"buy buy buy buy buy buy buy buy"),
 ("e","ru",.9,"Градиенты помогают обновлять параметры модели при обучении"),
 ("f","xx",.8,"unknown script placeholder text for this example"),
 ("g","en",.8,"Email example.person@example.test for the synthetic exercise"),
 ("h","en",.9,"attention reads values by comparing queries and keys"),
]
def words(text): return re.findall(r"\w+",text.casefold())
def shingles(text,k=2):
 w=words(text);return {tuple(w[i:i+k]) for i in range(len(w)-k+1)}
def jaccard(a,b): return len(a&b)/len(a|b) if a|b else 1.
def curate(order):
 kept=[];counts=Counter();seen=[]
 for docid,lang,quality,text in docs:
  failed=None
  for stage in order:
   if stage=="language" and lang not in {"en","ru"}:failed=stage
   elif stage=="quality" and quality<.5:failed=stage
   elif stage=="pii" and re.search(r"\S+@\S+",text):failed=stage
   elif stage=="dedup" and any(jaccard(shingles(text),s)>=.5 for s in seen):failed=stage
   if failed:break
  if failed:counts[failed]+=1
  else:kept.append(docid);seen.append(shingles(text))
 counts["kept"]=len(kept)
 assert sum(counts.values())==len(docs)
 return kept,counts
kept,counts=curate(["language","quality","pii","dedup"])
assert kept==["a","e","h"]
print("survival:",counts,"retained:",kept)
print("near-duplicate Jaccard:",jaccard(shingles(docs[0][3]),shingles(docs[2][3])))
# Cross-language paraphrases are not detected by this lexical mechanism.
assert jaccard(shingles(docs[0][3]),shingles(docs[4][3])) == 0
fig,ax=plt.subplots(figsize=(11,4.5),layout="constrained")
ax.bar(list(counts),list(counts.values()),color="#5E81AC")
ax.set_ylabel("Документы" if DEMO_LANGUAGE=="ru" else "Documents")
ax.set_title("Исключающие причины отсева" if DEMO_LANGUAGE=="ru" else "Exclusive retention/rejection counts")
plt.show()

The initial order retains a, e, and h. Mutually exclusive rejection counts plus retained documents sum to eight. Lexical matching cannot establish cross-language or semantic equivalence, and supplied quality labels do not test a classifier. Reordering filters may change the attributed causes even when the final retained set stays unchanged.

## Demo 2. Owned shards and temporary full weights

Split an 8×4 matrix by rows across two simulated ranks. Gather it for computation, average gradients from equally sized local batches, then return each gradient shard to its owner. An unsharded update supplies the reference. The later 1B-model memory ledger is a separate hypothetical AdamW policy, not a measurement of the float64 arrays used in this experiment.

In [ ]:
# Simulate two ranks on CPU: all-gather weights, reduce-scatter gradients,
# update owned shards. No actual network, FSDP runtime, or GPU memory benchmark.
rng=np.random.default_rng(SEED)
x=rng.normal(size=(12,8));y=rng.normal(size=(12,4));w=rng.normal(size=(8,4))
shards=[a.copy() for a in np.array_split(w,2,axis=0)]
lr=.03
for step in range(4):
 full=np.concatenate(shards,axis=0)
 # Equal local batch sizes; average local gradients gives the global gradient.
 local=[]
 for xb,yb in zip(np.array_split(x,2),np.array_split(y,2)):
  local.append(xb.T@(xb@full-yb)/len(xb))
 grad=sum(local)/2
 refgrad=x.T@(x@w-y)/len(x)
 np.testing.assert_allclose(grad,refgrad,atol=1e-12)
 for i,g in enumerate(np.array_split(grad,2,axis=0)):shards[i]-=lr*g
 w-=lr*refgrad
 np.testing.assert_allclose(np.concatenate(shards),w,atol=1e-12)
assert all(s.nbytes==w.nbytes//2 for s in shards)
print("CPU shard parity: PASS; owned weight bytes/rank:",shards[0].nbytes,
      "temporary gathered weights:",full.nbytes)
# Different ledger: hypothetical mixed-precision AdamW, not this NumPy dtype.
P=1_000_000_000
for world in (1,2,8):
 for stage in (0,1,2,3):
  # Per parameter: weights 2 B, gradients 2 B, master+moments 12 B.
  per_parameter=(16,4+12/world,2+14/world,16/world)[stage]
  payload=P*per_parameter
  print(f"ZeRO-{stage}, {world} ranks: {payload/1e9:.3f} GB/rank")
 # An illustrative additional gather window and live activation reserve.
 gather=100_000_000*2 if world>1 else 0
 activations=500_000_000
 with_buffers=P*16/world+gather+activations
 print(f"ZeRO-3 plus declared buffers: {with_buffers/1e9:.3f} GB/rank")
print("Activations, gathers, communication buffers, and allocator reserve are additional.")

Four-step parity checks gradient averaging and shard ownership. Each rank owns half the weights, yet computation temporarily materializes a full matrix. The separate ledger shards different components at different ZeRO stages. Its 0.2 GB gather window and 0.5 GB activation reserve are declared assumptions that do not divide by rank count, not a measured peak.

## Scaling to the full project

For E4, vary rule order and thresholds, explaining each changed outcome. Pin the revision, license, and sample size if you add an optional external corpus. For E5, vary shard count and buffer reserves. Keep local batches equal for simple averaging, or derive example-count weights for unequal batches. Distributed network execution is outside this exercise.

### Minimum report

1. Data, model, and code revisions, plus the random seed.
2. The primary metric, plus quality, cost, latency, and memory where applicable.
3. A baseline, a matched budget, and at least one negative control.
4. Failed runs and the limits of the supported conclusions.